# Synthea Exploratory Data Analysis (EDA) Template

Interactive notebook for exploring synthetic EHR datasets generated by [Synthea](https://github.com/synthetichealth/synthea) (`CSV` or `FHIR R4 JSON`).

This notebook reuses the loaders and helpers from `synthea_eda.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from synthea_eda import (
    detect_format,
    load_csv_tables,
    load_fhir_tables,
    summarize_tables,
    compute_patient_ages,
)

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 1. Configuration & Data Loading

In [ ]:
# Point INPUT_DIR to your Synthea output/csv or output/fhir directory
INPUT_DIR = Path("./output/csv")
DATA_FORMAT = "auto"  # "auto", "csv", or "fhir"
TOP_N = 15

fmt = detect_format(INPUT_DIR) if DATA_FORMAT == "auto" else DATA_FORMAT
tables = load_csv_tables(INPUT_DIR) if fmt == "csv" else load_fhir_tables(INPUT_DIR)
print(f"Loaded {len(tables)} tables in '{fmt}' format.")
summarize_tables(tables)

## 2. Patient Demographics

In [ ]:
patients = tables.get("patients", pd.DataFrame())
if not patients.empty:
    patients["AGE"] = compute_patient_ages(patients)
    display(patients.describe(include="all").T[["count", "unique", "top", "mean", "min", "max"]])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    sns.histplot(patients["AGE"].dropna(), bins=25, kde=True, ax=axes[0])
    axes[0].set_title("Age Distribution")

    if "GENDER" in patients.columns:
        patients["GENDER"].value_counts().plot(kind="bar", ax=axes[1], title="Gender")
    if "RACE" in patients.columns:
        patients["RACE"].value_counts().plot(kind="barh", ax=axes[2], title="Race")
    plt.tight_layout()
    plt.show()

## 3. Encounters & Utilization

In [ ]:
encounters = tables.get("encounters", pd.DataFrame())
if not encounters.empty:
    if "ENCOUNTERCLASS" in encounters.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        encounters["ENCOUNTERCLASS"].value_counts().plot(kind="barh", ax=ax, title="Encounters by Class")
        plt.tight_layout()
        plt.show()

    cost_cols = [c for c in ("BASE_ENCOUNTER_COST", "TOTAL_CLAIM_COST", "PAYER_COVERAGE") if c in encounters.columns]
    if cost_cols:
        display(encounters[cost_cols].apply(pd.to_numeric, errors="coerce").describe().round(2))

## 4. Top Clinical Events (Conditions, Medications, Procedures)

In [ ]:
for tname in ["conditions", "medications", "procedures", "allergies", "immunizations"]:
    df = tables.get(tname)
    if df is not None and not df.empty and "DESCRIPTION" in df.columns:
        top_df = df["DESCRIPTION"].value_counts().head(TOP_N)
        fig, ax = plt.subplots(figsize=(9, 4.5))
        top_df.sort_values().plot(kind="barh", ax=ax, title=f"Top {len(top_df)} {tname.title()}")
        plt.tight_layout()
        plt.show()

## 5. Observations (Vitals & Labs)

In [ ]:
obs = tables.get("observations", pd.DataFrame())
if not obs.empty and {"DESCRIPTION", "VALUE"}.issubset(obs.columns):
    obs["NUM_VALUE"] = pd.to_numeric(obs["VALUE"], errors="coerce")
    summary = (
        obs.dropna(subset=["NUM_VALUE"])
        .groupby("DESCRIPTION")["NUM_VALUE"]
        .agg(["count", "mean", "std", "min", "median", "max"])
        .sort_values("count", ascending=False)
        .head(15)
        .round(2)
    )
    display(summary)